## Libraries

In [ ]:
library(Seurat)
library(DESeq2)
library(readr)
library(dplyr)
library(tibble)
library(readxl)
library(pheatmap)
library(writexl)
library(ggplot2)
library(stringr)
library(openxlsx)
library(fgsea)
library(stringr)
library(tidyr)
library(tximport)
library(ape)

## **Processing Counts**

### Generate first metadata table

In [ ]:
# Define functions

getBaseName <- function(sample, condition) {
    # Concatenate sample and condition strings
    baseName = paste0(sample, '.', condition)
    baseName
}

getSeuratObj <- function(baseName) {
    # Import counts into a seurat object
    fName <- paste0('/rprojectnb/cancergrp/brb/intermediate_outputs/', baseName, '/out.Solo.out/Gene/raw')
    data <- Read10X(data.dir = fName)
    seurat_object <- CreateSeuratObject(counts = data)
    seurat_object
}

getMetadata <- function(baseName, seurat_object) {
    
    # Import .tsv file with information of each well
    fName = paste0('/rprojectnb/cancergrp/brb/metadataBarcodes/', baseName, '.metadata.tsv')
    metadata = read_tsv(fName, col_names = c('Well', 'Type', 'BC'), show_col_types = FALSE)

    col1 <- c()
    col2 <- c()
    rawcol1 <- c()
    rawcol2 <- c()
    totalcol <- c()

    # Iterate over wells and get matching values
    
    for (i in 1:nrow(metadata)){
        rowData = metadata[i,]
        Well = rowData[['Well']]
        Type = rowData[['Type']]
        BC = rowData[['BC']]
        
        n_total_counts = sum(seurat_object@assays$RNA$counts[, BC])

        # Last 84 values belong to vTR genes
        values <- seurat_object@assays$RNA$counts[(63224-83):63224, BC]
        
        raw_values <- sort(values, decreasing = TRUE)
        
        values <- round(sort(values, decreasing = TRUE)/ sum(values) *100, 2)
        
    
        if (Type %in% c('ctrl', 'vector')){
            bc_prop <- NA
            bc_raw <- NA
            names(bc_prop) <- Type
            names(bc_raw) <- Type
            
        } else{
            bc_prop <- values[Type]
            bc_raw <- raw_values[Type]
        }
        
        greatest_prop <- values[1]
        greatest_raw <- raw_values[1]
        col1 <- c(col1, bc_prop)
        col2 <- c(col2, greatest_prop)
        rawcol1 <- c(rawcol1, bc_raw)
        rawcol2 <- c(rawcol2, greatest_raw)
        totalcol <- c(totalcol, n_total_counts)
    }
    
    metadata$vtr_name = names(col1)
    metadata$vtr_prop = col1
    metadata$vtr_raw = rawcol1
    metadata$best_name = names(col2)
    metadata$best_prop = col2
    metadata$best_raw = rawcol2
    metadata$n_total_counts = totalcol
    metadata$match = ifelse(names(col1) == names(col2), "Yes", "No")
    metadata
}

getStatsFromMetadata <- function(metadata) {
    
    metadata_vTRs <- metadata[!metadata$vtr_name %in% c('ctrl', 'vector'),]

    # chosen cutoff was 85%
    wells_greater85 <- metadata_vTRs[metadata_vTRs$best_prop >= 85,]$Well
    
    # check if these correspond to their expected expressed vTR
    matching_vTRs <- metadata_vTRs[metadata_vTRs$vtr_name == metadata_vTRs$best_name,]$Well
    
    print(paste0(length(wells_greater85),"/",
                 nrow(metadata_vTRs),
                 " wells have >85% reads aligned to some vTR"))
    print(paste0(sum(wells_greater85 %in% matching_vTRs), "/", length(wells_greater85), " express the expected vTR."))

    list(wells_greater85, matching_vTRs, wells_greater85[wells_greater85 %in% matching_vTRs])
}

In [ ]:
# Define batches and conditions

batches <- paste0(rep("Batch", 12), rep(1:6, each=2))
conditions <- rep(c("unstimulated", "stimulated"), 6)

# Initialize metadata and stats objects

metadataList <- list()
statsDf <- data.frame(Batch=c(), Condition=c(), greater85=c(), matching=c(), match85=c())

# Iterate over batches/plates using metadata and retrieving stats

for (i in 1:length(batches)) {

    baseName <- getBaseName(batches[i], conditions[i])
    seuratObj <- getSeuratObj(baseName)
    metadata <- getMetadata(baseName, seuratObj)
    stats <- getStatsFromMetadata(metadata)

    metadataList[[baseName]] <- metadata

    batchStats <- data.frame(batches[i], conditions[i], length(stats[[1]]), length(stats[[2]]), length(stats[[3]]))
    colnames(batchStats) <- c("Batch", "Condition", "greater85", "matching", "match85")
    
    statsDf <- rbind(statsDf, batchStats)
    
}


In [ ]:
# Convertion of metadata list object into dataframe 'megaMetadata'

# Initialization of dataframe
megaMetadata <- data.frame(BatchID = c(),
                           "Stim/Unstim" = c(),
                           Well = c(), 
                           Expected_VTR = c(),
                           Proportion_of_reads_in_expected_VTR = c(),
                           n_reads_in_expected_VTR = c(),
                           top_VTR = c(),
                           Proportion_of_reads_in_top_VTR = c(),
                           n_reads_in_top_VTR = c(),
                           n_total_counts = c(),
                           Match = c())

# Iteration for each batch/plate
for (i in 1:length(metadataList)) {
    batch_id <- strsplit(names(metadataList)[i], "\\.")[[1]][1]
    condition <- strsplit(names(metadataList)[i], "\\.")[[1]][2]

    print(batch_id)
    
    metadata <- metadataList[[i]]

    batches <- rep(batch_id, nrow(metadata))
    condition <- rep(condition, nrow(metadata))

    metadata <- cbind(batches, condition, metadata[, !names(metadata) %in% c("BC", "Type")])

    names(metadata) <- names(megaMetadata)

    megaMetadata <- rbind(megaMetadata, metadata)
    
}

# Rename columns
names(megaMetadata) <- names(megaMetadata) <- c("BatchID", "Stim/Unstim", "Well", "Expected_VTR", "Proportion_of_reads_in_expected_VTR",
                                                "n_reads_in_expected_VTR", "top_VTR", "Proportion_of_reads_in_top_VTR", "n_reads_in_top_VTR", "n_total_counts",  "Match")

In [ ]:
# Aditional changes to megaMetadata dataframe (new stats and correction of Batch5)

## Adding prop of reads from all reads
megaMetadata$prop_top = megaMetadata$n_reads_in_top_VTR/megaMetadata$n_total_counts * 100

## Correcting some wells in Unstim area actually Stim
megaMetadata$`Stim/Unstim2` = megaMetadata$`Stim/Unstim`
megaMetadata$`Stim/Unstim2`[megaMetadata$BatchID == 'Batch5' & megaMetadata$Well %in% c('F11', 'F12', 'G09', 'G10', 'G11', 'G12', 'H09', 'H10', 'H11', 'H12')] = 'stimulated'

## New match filtering:
megaMetadata$Match2 = 'No'

## Parameters for VTR samples
cutoff_prop = 75
min_counts = 500000
megaMetadata$Match2[ !(megaMetadata$Expected_VTR %in% c('ctrl', 'vector')) & (megaMetadata$Proportion_of_reads_in_top_VTR > cutoff_prop) & (megaMetadata$n_total_counts >= min_counts)] = 'Yes'

## Parameters for control and vector samples 
cutoff_prop = 0.01
min_counts = 500000
megaMetadata$Match2[ (megaMetadata$Expected_VTR %in% c('ctrl', 'vector')) & (megaMetadata$prop_top < cutoff_prop) & (megaMetadata$n_total_counts >= min_counts)] = 'Yes'
megaMetadata$top_VTR[megaMetadata$Expected_VTR == 'ctrl'] = 'ctrl'
megaMetadata$top_VTR[megaMetadata$Expected_VTR == 'vector'] = 'vector'


# Create id: which is Batch + stim/unstim + well
megaMetadata = megaMetadata %>% mutate(id = paste(paste0(BatchID, '.', `Stim/Unstim`), Well, sep ='_')) 

# Create id_n: which is replicate number
megaMetadata = megaMetadata %>% group_by(BatchID, `Stim/Unstim2`, top_VTR) %>% mutate(id_n = row_number())

# Create id2: which is Batch + stim/unstim corrected + well + top_vtr
megaMetadata = megaMetadata %>% mutate(id2 = paste(paste0(BatchID, '.', `Stim/Unstim2`), Well, top_VTR,  id_n, sep ='_')) 

# Save megaMetadata dataframe as xlsx file
dir.create("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/", recursive = TRUE, showWarnings = FALSE)
write.xlsx(megaMetadata, '/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/brb_metadata.xlsx')